Testando GPU

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

Instalando bibliotecas

In [2]:
!pip install -q transformers accelerate scikit-learn pandas numpy fastapi uvicorn

Rodar code Clickbait_bertimbau.py

In [ ]:
!python /content/clickbait_bertimbau_v3.py train --csv "/content/novo dataset + noticias clickbait csv.csv"

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Classificado errado - Falso negativo

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_dir = "/content/saida_bertimbau/modelo_bertimbau_clickbait"

test_df = pd.read_csv(
    "/content/saida_bertimbau/split_test.csv",
    sep=";"
)

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

probabilidades = []

for titulo in test_df["titulo"]:
    inputs = tokenizer(
        titulo,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        prob = torch.softmax(logits, dim=1)[0][1].item()
        probabilidades.append(prob)

test_df["prob_clickbait"] = probabilidades

# Use o limiar do novo treinamento
test_df["predicao"] = (
    test_df["prob_clickbait"] >= 0.865
).astype(int)

erros = test_df[
    (test_df["clickbait_label_v2"] == 1) &
    (test_df["predicao"] == 0)
]

print("Quantidade:", len(erros))

resultado = erros[[
    "titulo",
    "clickbait_label_v2",
    "predicao",
    "prob_clickbait"
]]

display(resultado)

# Salvar em Excel
resultado.to_excel(
    "/content/clickbaits_nao_detectados.xlsx",
    index=False
)

print("Arquivo criado: clickbaits_nao_detectados.xlsx")

Teste manual de classificacao

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_dir = "/content/saida_bertimbau/modelo_bertimbau_clickbait"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

noticias = [
    "Passageiro indisciplinado pode ficar até um ano sem voar a partir de hoje",
    "5 sinais de que bots estão distorcendo o tráfego do seu negócio",
    "A IA escolhe o que vende nas redes? Veja 5 cuidados para as marcas",
    "André Mendonça retira sigilo de rede de pagamentos do Banco Master",
    "STF divulga rito da sessão que vai analisar relatório da PF que mostra suposta relação de Vorcaro e Moraes"
]

for titulo in noticias:
    inputs = tokenizer(
        titulo,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        prob = torch.softmax(logits, dim=1)[0][1].item()

    predicao = 1 if prob >= 0.865 else 0

    print("\nTítulo:", titulo)
    print("Probabilidade clickbait:", round(prob, 4))
    print("Classificação:", predicao)